# Constitutional AI: AI Feedback for Alignment

## Learning Objectives
1. Understand how constitutional principles guide AI alignment
2. Implement constitutional critique and revision loops
3. Analyze the self-improvement mechanism in Constitutional AI
4. Compare Constitutional AI with RLHF and DPO approaches

In [ ]:
import torch
import numpy as np
from typing import Optional, Dict, List
import json

# Device setup for reproducibility
np.random.seed(42)
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Level 1: Basic Constitutional Critique

The foundation of Constitutional AI is using explicit principles to generate feedback.

In [ ]:
def basic_constitutional_critique(
    response: str,
    principles: List[str]
) -> Dict[str, str]:
    """
    Simulate basic constitutional critique without calling an API.
    In practice, this would call Claude or another LM.
    """
    critique = f"Issues found in response relative to principles:\n"
    
    # Simple heuristic checks for demonstration
    for principle in principles:
        if 'helpful' in principle.lower():
            if len(response) < 20:
                critique += f"- Response is too brief; violates '{principle}'\n"
        if 'harmless' in principle.lower():
            if 'violence' in response.lower() or 'harm' in response.lower():
                critique += f"- Response mentions harm; violates '{principle}'\n"
        if 'honest' in principle.lower():
            if 'believe' in response.lower() and len(response) < 50:
                critique += f"- Response lacks depth; may violate '{principle}'\n"
    
    if critique == f"Issues found in response relative to principles:\n":
        critique += "No major issues found against stated principles."
    
    # Generate revision
    revision = response + " (revised to follow principles)"
    
    return {
        'original': response,
        'critique': critique,
        'revised': revision
    }

# Example usage
constitution = [
    'Be helpful and provide useful information',
    'Be harmless and avoid harmful content',
    'Be honest and acknowledge uncertainty'
]

prompt = 'What is machine learning?'
response = 'ML is cool.'

result = basic_constitutional_critique(response, constitution)
print(f'Prompt: {prompt}')
print(f'Original Response: {result["original"]}')
print(f'Critique:\n{result["critique"]}')
print(f'Revised: {result["revised"]}')

## Level 2: Advanced Constitutional Feedback System

A more sophisticated system that evaluates multiple responses and tracks alignment metrics.

In [ ]:
class ConstitutionalFeedbackSystem:
    """
    Advanced constitutional AI feedback system with multi-principle evaluation.
    """
    
    def __init__(self, principles: List[str]):
        """
        Initialize the system with constitutional principles.
        
        Args:
            principles: List of constitutional principles
        """
        self.principles = principles
        self.principle_weights = {p: 1.0 / len(principles) for p in principles}
        self.feedback_history = []
    
    def evaluate_against_principles(
        self,
        response: str
    ) -> Dict[str, float]:
        """
        Evaluate response against each principle.
        
        Args:
            response: Response to evaluate
        
        Returns:
            Dictionary mapping principle -> score (0-1)
        """
        scores = {}
        response_lower = response.lower()
        
        for principle in self.principles:
            if 'helpful' in principle.lower():
                # Score based on length and information density
                score = min(len(response.split()) / 20, 1.0)
            elif 'harmless' in principle.lower():
                # Score based on avoiding harmful content
                harmful_terms = ['violence', 'illegal', 'dangerous', 'harm']
                score = 1.0 - (sum(1 for term in harmful_terms if term in response_lower) * 0.25)
            elif 'honest' in principle.lower():
                # Score based on acknowledging uncertainty
                uncertain_terms = ['may', 'might', 'could', 'uncertain', 'unsure']
                score = 0.7 + 0.3 * min(sum(1 for term in uncertain_terms if term in response_lower), 3) / 3
            else:
                score = 0.5  # Default score
            
            scores[principle] = max(0.0, min(1.0, score))
        
        return scores
    
    def compute_overall_alignment(
        self,
        principle_scores: Dict[str, float]
    ) -> float:
        """
        Compute weighted overall alignment score.
        
        Args:
            principle_scores: Scores for each principle
        
        Returns:
            Overall alignment score (0-1)
        """
        total_score = sum(
            principle_scores[p] * self.principle_weights[p]
            for p in self.principles
        )
        return total_score
    
    def generate_critique(
        self,
        response: str,
        principle_scores: Dict[str, float]
    ) -> str:
        """
        Generate detailed critique based on principle scores.
        
        Args:
            response: Original response
            principle_scores: Evaluation scores
        
        Returns:
            Critique text
        """
        critique = "Constitutional Critique:\n"
        
        for principle, score in principle_scores.items():
            status = 'Strong' if score > 0.8 else 'Moderate' if score > 0.5 else 'Weak'
            critique += f"- {principle}: {status} ({score:.2f})\n"
        
        return critique
    
    def process_response(
        self,
        response: str
    ) -> Dict:
        """
        Full processing pipeline for a response.
        
        Args:
            response: Response to process
        
        Returns:
            Dictionary with scores and critique
        """
        scores = self.evaluate_against_principles(response)
        overall = self.compute_overall_alignment(scores)
        critique = self.generate_critique(response, scores)
        
        result = {
            'response': response,
            'principle_scores': scores,
            'overall_alignment': overall,
            'critique': critique
        }
        
        self.feedback_history.append(result)
        return result

# Initialize system
advanced_constitution = [
    'Be helpful: provide accurate, useful information',
    'Be harmless: avoid harmful, illegal, or dangerous content',
    'Be honest: acknowledge limitations and uncertainty'
]

system = ConstitutionalFeedbackSystem(advanced_constitution)

# Test with different responses
test_responses = [
    'Machine learning is when computers learn from data.',
    'ML is AI. ML can do anything. Very powerful.',
    'Machine learning is a subset of AI that learns patterns from data. It\'s powerful but has limitations. I\'m uncertain about its full potential.'
]

for resp in test_responses:
    result = system.process_response(resp)
    print(f'Response: {resp}')
    print(f'Overall Alignment: {result["overall_alignment"]:.2f}')
    print(result['critique'])
    print()

## Real-World Example 1: Constitutional Feedback for Question-Answering

Apply constitutional evaluation to a Q&A system with iterative improvement.

In [ ]:
class IterativeConstitutionalImprovement:
    """
    Simulate iterative response improvement based on constitutional feedback.
    """
    
    def __init__(self, constitution: List[str]):
        self.constitution = constitution
        self.system = ConstitutionalFeedbackSystem(constitution)
        self.improvement_history = []
    
    def improve_response(self, response: str, num_iterations: int = 3) -> List[Dict]:
        """
        Iteratively improve a response based on constitutional feedback.
        
        Args:
            response: Initial response to improve
            num_iterations: Number of improvement iterations
        
        Returns:
            List of improvement steps with scores
        """
        current_response = response
        history = []
        
        for iteration in range(num_iterations):
            # Evaluate current response
            feedback = self.system.process_response(current_response)
            
            history.append({
                'iteration': iteration,
                'response': current_response,
                'alignment_score': feedback['overall_alignment'],
                'principle_scores': feedback['principle_scores']
            })
            
            # Simulate improvement based on weakest principle
            weakest_principle = min(
                feedback['principle_scores'].items(),
                key=lambda x: x[1]
            )[0]
            
            if 'helpful' in weakest_principle.lower():
                current_response += ' Here\'s more detail: [additional context]'
            elif 'harmless' in weakest_principle.lower():
                current_response = current_response.replace('danger', 'risk')
            elif 'honest' in weakest_principle.lower():
                if 'uncertain' not in current_response.lower():
                    current_response += ' However, I acknowledge uncertainty in some aspects.'
        
        self.improvement_history.append(history)
        return history

# Example: Improve a response iteratively
improver = IterativeConstitutionalImprovement(advanced_constitution)

initial_response = 'AI is very powerful.'
improvement_steps = improver.improve_response(initial_response, num_iterations=3)

print('Iterative Constitutional Improvement:')
for step in improvement_steps:
    print(f"\nIteration {step['iteration']}:")
    print(f"  Response: {step['response'][:80]}...")
    print(f"  Overall Alignment: {step['alignment_score']:.3f}")
    print(f"  Per-principle scores:")
    for principle, score in step['principle_scores'].items():
        print(f"    - {principle[:30]}...: {score:.3f}")

## Real-World Example 2: Multi-Response Constitutional Selection

Generate multiple candidates and select the best based on constitutional principles.

In [ ]:
class ConstitutionalResponseSelector:
    """
    Generate and rank multiple responses using constitutional principles.
    """
    
    def __init__(self, constitution: List[str]):
        self.constitution = constitution
        self.system = ConstitutionalFeedbackSystem(constitution)
    
    def generate_candidate_responses(self, prompt: str, num_candidates: int = 3) -> List[str]:
        """
        Simulate generation of multiple candidate responses.
        In practice, these would come from model sampling.
        """
        candidates = [
            f'Response A to prompt about {prompt}',
            f'An alternative perspective on {prompt} is interesting.',
            f'Regarding {prompt}, this is complex. I have some thoughts but also acknowledge uncertainty.'
        ]
        return candidates[:num_candidates]
    
    def rank_responses(self, candidates: List[str]) -> List[Dict]:
        """
        Rank candidates by constitutional alignment.
        
        Args:
            candidates: List of candidate responses
        
        Returns:
            Sorted list of candidates with scores
        """
        scored_candidates = []
        
        for candidate in candidates:
            feedback = self.system.process_response(candidate)
            scored_candidates.append({
                'response': candidate,
                'alignment_score': feedback['overall_alignment'],
                'principle_scores': feedback['principle_scores']
            })
        
        # Sort by alignment score descending
        return sorted(scored_candidates, key=lambda x: x['alignment_score'], reverse=True)
    
    def select_best(self, prompt: str, num_candidates: int = 3) -> Dict:
        """
        Generate and select the best response according to constitution.
        
        Args:
            prompt: Input prompt
            num_candidates: Number of candidates to generate
        
        Returns:
            Best response and ranking
        """
        candidates = self.generate_candidate_responses(prompt, num_candidates)
        ranked = self.rank_responses(candidates)
        
        return {
            'prompt': prompt,
            'best_response': ranked[0]['response'],
            'best_score': ranked[0]['alignment_score'],
            'ranking': ranked
        }

# Example: Select best response from candidates
selector = ConstitutionalResponseSelector(advanced_constitution)

prompt = 'What is the future of AI?'
result = selector.select_best(prompt, num_candidates=3)

print(f"Prompt: {result['prompt']}")
print(f"\nBest Response (score: {result['best_score']:.3f}):")
print(f"  {result['best_response']}")
print(f"\nRanking of all candidates:")
for i, candidate in enumerate(result['ranking']):
    print(f"  {i+1}. Score: {candidate['alignment_score']:.3f}")
    print(f"     {candidate['response'][:70]}...")

## Key Takeaways

**Core idea:** Constitutional AI uses explicit principles to guide model alignment without human feedback on each decision.

**Approaches and when to use:**
| Method | Use when | Key advantage |
|--------|----------|---------------|
| Critique & revision | You want iterative improvement | Self-improving feedback loop |
| Response selection | Choosing among candidates | Scale to many candidate evaluations |
| Iterative refinement | Principles need multiple passes | Compound improvements |
| Multi-principle weighting | Different goals matter | Balanced alignment across objectives |

**Common failure modes and fixes:**
- Principles too vague: Model doesn't know what "helpful" means specifically
  → Fix: Include concrete examples and specific criteria
- AI feedback doesn't match human values: Model learns wrong patterns
  → Fix: Validate with human evaluation on sample of critiques
- Responses become repetitive/formulaic: Over-optimization for principles
  → Fix: Add diversity metrics, avoid extremely high principle scores

**Related concepts:**
- [RLHF/InstructGPT](../concepts/03-rlhf-instructgpt.ipynb) – Human feedback with explicit reward models
- [DPO](../concepts/02-dpo.ipynb) – Direct preference optimization without reward models

## Exercises

1. **Modify constitutional principles**: Change the principles to emphasize safety over helpfulness. How does the ranking of responses change?

2. **Test on edge cases**: What happens when you apply constitutional evaluation to ambiguous or sensitive prompts? Where does it break?

3. **Add new principle**: Extend the system to evaluate "truthfulness" by checking if responses make specific factual claims (they shouldn't without evidence).

4. **Compare approaches**: How does Constitutional AI's simplicity compare to RLHF? When would each be preferred?